# 쇼핑몰 DB CRUD — 회원, 상품, 주문

**사전 준비:**
1. `.env.example`을 복사하여 `.env`를 만들고 Supabase URL과 Key를 입력합니다.
2. Supabase SQL Editor에서 아래 SQL 스키마를 실행하여 테이블을 먼저 생성합니다.

## 0. 테이블 생성 SQL (Supabase SQL Editor에서 실행)

```sql
-- 1. 회원 테이블
CREATE TABLE members (
    member_id   SERIAL PRIMARY KEY,
    username    VARCHAR(50)  UNIQUE NOT NULL,
    email       VARCHAR(100) UNIQUE NOT NULL,
    full_name   VARCHAR(100) NOT NULL,
    phone       VARCHAR(20),
    role        VARCHAR(10)  NOT NULL DEFAULT 'customer'
                    CHECK (role IN ('customer', 'admin', 'seller')),
    created_at  TIMESTAMPTZ  NOT NULL DEFAULT NOW()
);

-- 2. 상품 테이블
CREATE TABLE products (
    product_id  SERIAL PRIMARY KEY,
    name        VARCHAR(200) NOT NULL,
    description TEXT,
    price       NUMERIC(12,2) NOT NULL CHECK (price >= 0),
    stock       INTEGER       NOT NULL DEFAULT 0 CHECK (stock >= 0),
    category    VARCHAR(50),
    is_active   BOOLEAN       NOT NULL DEFAULT TRUE,
    created_at  TIMESTAMPTZ   NOT NULL DEFAULT NOW()
);

-- 3. 주문 테이블
CREATE TABLE orders (
    order_id         SERIAL PRIMARY KEY,
    member_id        INTEGER       NOT NULL REFERENCES members(member_id) ON DELETE RESTRICT,
    product_id       INTEGER       NOT NULL REFERENCES products(product_id) ON DELETE RESTRICT,
    quantity         INTEGER       NOT NULL CHECK (quantity > 0),
    total_amount     NUMERIC(12,2) NOT NULL CHECK (total_amount >= 0),
    status           VARCHAR(20)   NOT NULL DEFAULT 'pending'
                         CHECK (status IN ('pending','confirmed','shipped','delivered','cancelled')),
    shipping_address TEXT          NOT NULL,
    created_at       TIMESTAMPTZ   NOT NULL DEFAULT NOW()
);
```

## 1. 환경 설정 및 Supabase 연결

In [ ]:
import os
from dotenv import load_dotenv
from supabase import create_client, Client

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError(".env 파일에 SUPABASE_URL 과 SUPABASE_KEY 를 설정하세요.")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase 연결 성공")

## 2. 회원(members) CRUD

In [ ]:
# --- CREATE ---
new_members = [
    {"username": "kim_customer", "email": "kim@example.com", "full_name": "김고객", "phone": "010-1111-2222", "role": "customer"},
    {"username": "lee_seller",   "email": "lee@example.com", "full_name": "이판매자", "role": "seller"},
    {"username": "admin_park",   "email": "park@example.com", "full_name": "박관리자", "role": "admin"},
]
res_insert = supabase.table("members").insert(new_members).execute()
print("[CREATE] 회원 삽입:", res_insert.data)

In [ ]:
# --- READ (전체 조회) ---
res_all = supabase.table("members").select("*").execute()
print("[READ] 전체 회원:")
for m in res_all.data:
    print(f"  id={m['member_id']}  {m['full_name']}  ({m['role']})")

In [ ]:
# --- READ (조건 조회 — customer만) ---
res_filtered = supabase.table("members").select("*").eq("role", "customer").execute()
print("[READ] customer 회원:", res_filtered.data)

In [ ]:
# --- UPDATE ---
res_update = (
    supabase.table("members")
    .update({"phone": "010-9999-0000"})
    .eq("username", "lee_seller")
    .execute()
)
print("[UPDATE] lee_seller 전화번호 변경:", res_update.data)

In [ ]:
# --- DELETE ---
res_delete = (
    supabase.table("members")
    .delete()
    .eq("username", "admin_park")
    .execute()
)
print("[DELETE] admin_park 삭제:", res_delete.data)

# 삭제 후 전체 조회로 검증
res_after = supabase.table("members").select("username").execute()
print("[검증] 삭제 후 회원:", [r['username'] for r in res_after.data])

## 3. 상품(products) CRUD

In [ ]:
# --- CREATE ---
new_products = [
    {"name": "무선 이어폰",   "price": 89000,  "stock": 50,  "category": "electronics", "description": "노이즈 캔슬링 이어폰"},
    {"name": "겨울 패딩 점퍼", "price": 189000, "stock": 30,  "category": "clothing",     "description": "따뜻한 오리털 패딩"},
    {"name": "유기농 쌀 5kg", "price": 32000,  "stock": 200, "category": "food",          "description": "국내산 유기농 쌀"},
]
res_p = supabase.table("products").insert(new_products).execute()
print("[CREATE] 상품 삽입:", [p['name'] for p in res_p.data])

In [ ]:
# --- READ ---
res_p_all = supabase.table("products").select("*").order("price", desc=False).execute()
print("[READ] 상품 목록 (가격 오름차순):")
for p in res_p_all.data:
    print(f"  {p['name']:15s}  {p['price']:>10,}원  재고:{p['stock']}")

In [ ]:
# --- UPDATE: 재고 감소 ---
product_id = res_p.data[0]['product_id']  # 첫 번째 상품
res_stock = (
    supabase.table("products")
    .update({"stock": 45})
    .eq("product_id", product_id)
    .execute()
)
print("[UPDATE] 재고 변경:", res_stock.data)

In [ ]:
# --- DELETE: 비활성 처리 (soft delete) ---
res_soft = (
    supabase.table("products")
    .update({"is_active": False})
    .eq("name", "유기농 쌀 5kg")
    .execute()
)
print("[SOFT DELETE] 비활성화:", res_soft.data)

## 4. 주문(orders) CRUD

In [ ]:
# 주문 생성에 필요한 ID 조회
members = supabase.table("members").select("member_id, username").execute().data
products = supabase.table("products").select("product_id, name, price").eq("is_active", True).execute().data

print("회원:", [(m['member_id'], m['username']) for m in members])
print("상품:", [(p['product_id'], p['name'], p['price']) for p in products])

In [ ]:
# --- CREATE ---
member_id = members[0]['member_id']
product_id = products[0]['product_id']
unit_price = float(products[0]['price'])
qty = 2

new_order = {
    "member_id":        member_id,
    "product_id":       product_id,
    "quantity":         qty,
    "total_amount":     unit_price * qty,
    "shipping_address": "서울시 강남구 테헤란로 123",
    "status":           "pending",
}
res_o = supabase.table("orders").insert(new_order).execute()
print("[CREATE] 주문 생성:", res_o.data)

In [ ]:
# --- READ: 주문 + 회원/상품 JOIN ---
res_o_all = (
    supabase.table("orders")
    .select("order_id, status, total_amount, members(full_name), products(name)")
    .execute()
)
print("[READ] 주문 목록:")
for o in res_o_all.data:
    print(f"  주문#{o['order_id']}  {o['members']['full_name']}  {o['products']['name']}  {o['total_amount']}원  상태:{o['status']}")

In [ ]:
# --- UPDATE: 주문 상태 변경 ---
order_id = res_o.data[0]['order_id']
res_o_update = (
    supabase.table("orders")
    .update({"status": "confirmed"})
    .eq("order_id", order_id)
    .execute()
)
print("[UPDATE] 주문 확인:", res_o_update.data)

In [ ]:
# --- DELETE: 취소 주문 삭제 ---
# 먼저 상태를 cancelled로 변경 후 삭제
supabase.table("orders").update({"status": "cancelled"}).eq("order_id", order_id).execute()
res_o_del = supabase.table("orders").delete().eq("order_id", order_id).execute()
print("[DELETE] 주문 삭제:", res_o_del.data)

## 5. 트랜잭션 정합성 제어 (COMMIT / ROLLBACK)

Supabase Python SDK는 현재 직접적인 트랜잭션 API를 제공하지 않으므로,  
PostgreSQL Function(RPC)을 통해 트랜잭션을 구현합니다.

**Supabase SQL Editor에서 먼저 실행:**
```sql
CREATE OR REPLACE FUNCTION place_order(
    p_member_id  INT,
    p_product_id INT,
    p_quantity   INT,
    p_address    TEXT
) RETURNS JSONB
LANGUAGE plpgsql
AS $$
DECLARE
    v_price      NUMERIC;
    v_stock      INT;
    v_order_id   INT;
BEGIN
    -- 재고 조회
    SELECT price, stock INTO v_price, v_stock
    FROM products WHERE product_id = p_product_id FOR UPDATE;

    -- 재고 부족 시 ROLLBACK
    IF v_stock < p_quantity THEN
        RAISE EXCEPTION '재고 부족: 현재 재고 %개, 요청 수량 %개', v_stock, p_quantity;
    END IF;

    -- 재고 차감
    UPDATE products SET stock = stock - p_quantity WHERE product_id = p_product_id;

    -- 주문 삽입
    INSERT INTO orders (member_id, product_id, quantity, total_amount, shipping_address)
    VALUES (p_member_id, p_product_id, p_quantity, v_price * p_quantity, p_address)
    RETURNING order_id INTO v_order_id;

    RETURN jsonb_build_object('order_id', v_order_id, 'total', v_price * p_quantity);
END;
$$;
```

In [ ]:
# 트랜잭션 성공 시나리오 — RPC 호출
members = supabase.table("members").select("member_id").limit(1).execute().data
products = supabase.table("products").select("product_id, stock").eq("is_active", True).limit(1).execute().data

try:
    result = supabase.rpc(
        "place_order",
        {
            "p_member_id":  members[0]['member_id'],
            "p_product_id": products[0]['product_id'],
            "p_quantity":   1,
            "p_address":    "서울시 마포구 합정동 456",
        }
    ).execute()
    print("[COMMIT] 주문 성공:", result.data)
except Exception as e:
    print("[ROLLBACK] 주문 실패:", e)

In [ ]:
# 트랜잭션 실패 시나리오 — 재고보다 많은 수량 요청
current_stock = supabase.table("products").select("stock").eq("product_id", products[0]['product_id']).execute().data[0]['stock']
print(f"현재 재고: {current_stock}개 → 초과 수량({current_stock + 100}개) 주문 시도")

try:
    result = supabase.rpc(
        "place_order",
        {
            "p_member_id":  members[0]['member_id'],
            "p_product_id": products[0]['product_id'],
            "p_quantity":   current_stock + 100,  # 재고 초과
            "p_address":    "서울시 송파구 올림픽로 789",
        }
    ).execute()
    print("[COMMIT]", result.data)
except Exception as e:
    print("[ROLLBACK] 예상된 실패 — 재고 정합성 유지됨:", e)

# 재고가 그대로인지 검증
stock_after = supabase.table("products").select("stock").eq("product_id", products[0]['product_id']).execute().data[0]['stock']
print(f"롤백 후 재고 확인: {stock_after}개 (변화 없음: {current_stock == stock_after})")